# CSPF — stage F: severity × position × family, and flip rate by human label
Answers: (6) is there an *independent* effect of severity and of position, or are they nested in family? (3) flip rate for pairs the annotators labelled meaning-preserving / partial / meaning-altering, with CIs.
- **F1** counts and flip rate for every family × severity × position cell (cluster-bootstrap CI over source reviews).
- **F2** severity effect *within* family, with review fixed effects, position fixed effects, and only cells where all three severities exist (≥ MIN_CELL pairs each).
- **F3** position effect within family and severity, same adjustments.
- **F4** flip rate by human label (annotator 1, annotator 2, both) incl. 'partial' and 'meaning-altering', per-model values and Cochran's Q on the consensus-verified pairs.
CPU only, ~10 min. Needs `ALL_MODELS_combined_FIXED.csv`, `annotated_output.csv`, `annotation_sheet_for_second_annotator_FILLED.csv`. Output: `CSPF_stageF_bundle.zip`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"; HUMAN_A1 = f"{BASE}/annotated_output.csv"; HUMAN_A2 = f"{BASE}/annotation_sheet_for_second_annotator_FILLED.csv"
OUT = f"{BASE}/results_FIXED/reviewer_F"; MIN_CELL = 100; B_REP = 500
import os, subprocess, sys, warnings, zipfile, glob; warnings.filterwarnings("ignore"); os.makedirs(OUT, exist_ok=True)
subprocess.run([sys.executable,"-m","pip","-q","install","statsmodels"],check=False)
import numpy as np, pandas as pd
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
meta = ["source_dataset","review_idx","perturbed_review","perturbation_family","severity","position_strategy"]
use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
parts, Fl = [], []
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    Fl.append(np.stack([(ch[f"{m}_orig_label"].values != ch[f"{m}_pert_label"].values) for m in MODELS], 1).astype(np.int8)); parts.append(ch[meta])
df = pd.concat(parts, ignore_index=True); F = np.concatenate(Fl); del parts, Fl
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str); df["y"] = F.mean(1); df["sev"] = df.severity.astype(str)
kcodes, kinv = np.unique(df.key.values, return_inverse=True); NK = len(kcodes)
print(len(df), "pairs;", NK, "reviews;", len(MODELS), "models")

In [ ]:
# ── F1: family x severity x position ──
rng = np.random.default_rng(0)
cell = (df.perturbation_family+"|"+df.sev+"|"+df.position_strategy.astype(str)).values; ccodes, cinv = np.unique(cell, return_inverse=True)
n = np.bincount(cinv); est = np.bincount(cinv, weights=df.y.values)/n
bs = np.zeros((B_REP, len(ccodes)))
for b in range(B_REP):
    c = np.bincount(rng.integers(0, NK, NK), minlength=NK)[kinv].astype(float)
    bs[b] = np.bincount(cinv, weights=c*df.y.values, minlength=len(ccodes))/np.maximum(np.bincount(cinv, weights=c, minlength=len(ccodes)), 1e-9)
T = pd.DataFrame([c.split("|") for c in ccodes], columns=["family","severity","position"]); T["pairs"] = n; T["FR"] = est.round(4)
T["ci_lo"] = np.percentile(bs, 2.5, axis=0).round(4); T["ci_hi"] = np.percentile(bs, 97.5, axis=0).round(4)
T.to_csv(f"{OUT}/F1_family_severity_position.csv", index=False); print(T.groupby("family").size().to_dict(), "cells"); print((T.pairs < MIN_CELL).sum(), "cells below", MIN_CELL, "pairs")

In [ ]:
# ── F2: severity within family (review FE + position FE; only positions present at all three severities with >= MIN_CELL pairs each) ──
import statsmodels.api as sm
def within(Xdf, y, g):
    gi = pd.factorize(g)[0]; cnt = np.bincount(gi)
    mean = lambda v: np.bincount(gi, weights=v)/cnt
    Xd = np.stack([Xdf[c].values - mean(Xdf[c].values)[gi] for c in Xdf.columns], 1); return Xd, y - mean(y)[gi]
SEV_ORDER = [s for s in ["mild","medium","strong"] if s in set(df.sev)]; print("severity levels:", SEV_ORDER)
rows3, rowsP = [], []
for f, g in df.groupby("perturbation_family"):
    # F2
    cnt = g.groupby(["position_strategy","sev"]).size().unstack(fill_value=0)
    ok = cnt.index[(cnt[SEV_ORDER] >= MIN_CELL).all(axis=1)] if all(s in cnt.columns for s in SEV_ORDER) else []
    h = g[g.position_strategy.isin(ok)]
    if len(ok) == 0: rows3.append(dict(family=f, positions_kept=0, note="no position present at all severities")); continue
    X = pd.get_dummies(h.sev, drop_first=False).astype(float)[SEV_ORDER[1:]]
    for p in ok[1:]: X["pos_"+p] = (h.position_strategy.values == p).astype(float)
    Xd, yd = within(X, h.y.values, h.key.values); r = sm.OLS(yd, Xd).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(h.key.values)[0]})
    for i, s in enumerate(SEV_ORDER[1:]): rows3.append(dict(family=f, contrast=f"{s} vs {SEV_ORDER[0]}", positions_kept=len(ok), pairs=len(h), reviews=h.key.nunique(), coef=round(r.params[i],4), ci_lo=round(r.conf_int()[i,0],4), ci_hi=round(r.conf_int()[i,1],4)))
    # F3: position within family x severity (reference = most frequent position; positions with >= 500 pairs; severity FE)
    pc = g.position_strategy.value_counts(); keep = pc.index[pc >= 500]
    if len(keep) < 2: continue
    h = g[g.position_strategy.isin(keep)]; ref = keep[0]
    X = pd.concat([pd.get_dummies(h.sev).astype(float)[SEV_ORDER[1:]], pd.DataFrame({"pos_"+p: (h.position_strategy.values == p).astype(float) for p in keep[1:]}, index=h.index)], axis=1)
    Xd, yd = within(X, h.y.values, h.key.values); r = sm.OLS(yd, Xd).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(h.key.values)[0]})
    for i, c in enumerate(X.columns):
        if c.startswith("pos_"): rowsP.append(dict(family=f, position=c[4:], reference=ref, pairs=len(h), coef=round(r.params[i],4), ci_lo=round(r.conf_int()[i,0],4), ci_hi=round(r.conf_int()[i,1],4)))
S3 = pd.DataFrame(rows3); S3.to_csv(f"{OUT}/F2_severity_within_family.csv", index=False); print(S3.to_string(index=False))
P3 = pd.DataFrame(rowsP); P3.to_csv(f"{OUT}/F3_position_within_family.csv", index=False); print(P3.to_string(index=False))

In [ ]:
# ── F4: flip rate by human label ──
a1 = pd.read_csv(HUMAN_A1, encoding="utf-8-sig"); a2 = pd.read_csv(HUMAN_A2, encoding="utf-8-sig"); a2["meaning_a2"] = a2.meaning_a2.astype(str).str.strip().str.lower()
h = a1.merge(a2[["sample_id","meaning_a2"]], on="sample_id")
key = ["source_dataset","review_idx","perturbed_review","perturbation_family"]
h["source_dataset"] = h.source_dataset.astype(str)
lk = df[key+["key"]].assign(pair_id=np.arange(len(df))).drop_duplicates(key); j = h.merge(lk, on=key, how="inner"); print(len(j), "of", len(h), "audited pairs matched")
Fi = F[j.pair_id.values]; m1, m2 = j.meaning_preserving.values, j.meaning_a2.values
groups = {"annotator 1: yes": m1=="yes", "annotator 1: partial": m1=="partial", "annotator 1: no": m1=="no",
          "annotator 2: yes": m2=="yes", "annotator 2: partial": m2=="partial", "annotator 2: no": m2=="no",
          "both yes (consensus)": (m1=="yes")&(m2=="yes"), "either yes": (m1=="yes")|(m2=="yes"),
          "both partial": (m1=="partial")&(m2=="partial"), "either no": (m1=="no")|(m2=="no"), "all matched": np.ones(len(j), bool)}
uk, ki = np.unique(j.key.values, return_inverse=True); rows = []
for nm, m in groups.items():
    if m.sum() == 0: continue
    v = []
    for _ in range(1000):
        c = np.bincount(rng.integers(0, len(uk), len(uk)), minlength=len(uk))[ki][m]
        if c.sum(): v.append((c*Fi[m].mean(1)).sum()/c.sum())
    rows.append(dict(group=nm, pairs=int(m.sum()), FR=round(Fi[m].mean(),4), ci_lo=round(np.percentile(v,2.5),4), ci_hi=round(np.percentile(v,97.5),4)))
R4 = pd.DataFrame(rows); print(R4.to_string(index=False)); R4.to_csv(f"{OUT}/F4_FR_by_human_label.csv", index=False)
pd.DataFrame({"model": MODELS, **{nm: Fi[m].mean(0).round(4) for nm, m in groups.items() if m.sum() >= 30}}).to_csv(f"{OUT}/F4_per_model.csv", index=False)
from statsmodels.stats.contingency_tables import cochrans_q
mc = groups["both yes (consensus)"]; q = cochrans_q(Fi[mc]); print("Cochran Q on consensus-verified pairs:", q)
from scipy.stats import spearmanr; full = F.mean(0)
print("Spearman of per-model FR (consensus-verified vs full corpus):", spearmanr(Fi[mc].mean(0), full)[0])
pd.Series(dict(Q=q.statistic, df=q.df, p=q.pvalue, spearman_vs_full=spearmanr(Fi[mc].mean(0), full)[0])).to_csv(f"{OUT}/F4_consensus_tests.csv")
with zipfile.ZipFile(f"{OUT}/CSPF_stageF_bundle.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv"): z.write(f, os.path.basename(f))
print("send me:", f"{OUT}/CSPF_stageF_bundle.zip")